# GEO5017 — Waste Detection Pipeline
### Transfer Learning with ResNet-50

This notebook implements a full machine learning pipeline for urban waste detection
from streetview imagery, with safeguards for reproducibility, label validation,
ranking-oriented validation metrics, and leaderboard submission generation.

**Data split (as prescribed by the course):**
- Training:   year_2016, year_2017, year_2018, year_2019
- Validation: year_2020, year_2021
- Test:       year_2022, year_2023

**Pipeline steps:**
1. Setup & imports
2. Dataset preparation (rebuild-safe + label validation)
3. Data loaders with augmentation
4. Model definition (ResNet-50 with custom head)
5. Training with validation monitoring (Average Precision + P@100)
6. Fine-tuning (progressive unfreezing)
7. Evaluation on test set
8. Leaderboard: Top-100 detections


## 1. Setup & Imports

In [ ]:
import os
import shutil
import random
import json
import csv
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    average_precision_score
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


## 2. Configuration

All key paths and hyperparameters are defined in one place so they are easy to change.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
# Root folder containing year_2016 ... year_2023 subfolders
RAW_DATA_ROOT = Path('UrbanWaste-images-10k-right')

# Label file. Supported formats:
#   filename,label
#   relative_path,label
LABEL_CSV = Path('labels.csv')

# Restructured dataset folder (created by the dataset preparation step)
DATASET_ROOT = Path('dataset')

# Where to save model checkpoints
CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ── Data split by year ───────────────────────────────────────────────────────
TRAIN_YEARS = ['year_2016', 'year_2017', 'year_2018', 'year_2019']
VAL_YEARS   = ['year_2020', 'year_2021']
TEST_YEARS  = ['year_2022', 'year_2023']

# ── Dataset build options ────────────────────────────────────────────────────
REBUILD_DATASET = True   # If True, dataset/ is deleted and rebuilt from scratch
TOP_K           = 100    # Used for validation P@K and leaderboard generation

# ── Class-imbalance options ──────────────────────────────────────────────────
# Start with ONE imbalance strategy only. The default is the sampler.
USE_WEIGHTED_SAMPLER = True
USE_POS_WEIGHT       = False

if USE_WEIGHTED_SAMPLER and USE_POS_WEIGHT:
    raise ValueError(
        'Enable either USE_WEIGHTED_SAMPLER or USE_POS_WEIGHT, not both.'
    )

# ── Hyperparameters ──────────────────────────────────────────────────────────
IMG_SIZE        = 224      # ResNet standard input size
BATCH_SIZE      = 32
NUM_WORKERS     = 4        # Set to 0 on platforms without multiprocessing support

# Phase 1: only the new classification head is trained
LR_HEAD         = 1e-3
EPOCHS_HEAD     = 10

# Phase 2: classification head + last ResNet block (layer4) fine-tuned
LR_FINETUNE     = 1e-4
EPOCHS_FINETUNE = 10

# Early stopping: stop if val AP does not improve for this many epochs
PATIENCE        = 5

# ImageNet normalisation statistics (required because we use a pretrained model)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


## 3. Dataset Preparation

PyTorch's `ImageFolder` loader expects the following structure:

```
dataset/
├── train/
│   ├── waste/
│   └── no_waste/
├── val/
│   ├── waste/
│   └── no_waste/
└── test/
    ├── waste/
    └── no_waste/
```

The function below copies images from the year-based folders into this structure,
using a label CSV file that maps each filename to its class.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
VALID_LABELS = {'waste', 'no_waste'}


def _list_image_files(folder: Path) -> List[Path]:
    """Returns all supported image files in a folder, sorted for reproducibility."""
    return sorted(
        [
            p for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        ]
    )


def _normalise_label(label: str) -> str:
    label = str(label).strip().lower()
    if label not in VALID_LABELS:
        raise ValueError(
            f"Invalid label '{label}'. Expected one of {sorted(VALID_LABELS)}."
        )
    return label


def _discover_duplicate_filenames(raw_root: Path) -> Dict[str, List[str]]:
    """
    Scans the raw data tree for duplicated basenames across year folders.

    Returns
    -------
    dict mapping filename -> list of relative paths, only for duplicates.
    """
    seen = defaultdict(list)
    for year_dir in sorted([p for p in raw_root.iterdir() if p.is_dir()]):
        for img_path in _list_image_files(year_dir):
            seen[img_path.name].append(img_path.relative_to(raw_root).as_posix())

    return {
        fname: rel_paths
        for fname, rel_paths in seen.items()
        if len(rel_paths) > 1
    }


def _label_key_for_path(img_path: Path, raw_root: Path, key_column: str) -> str:
    if key_column == 'relative_path':
        return img_path.relative_to(raw_root).as_posix()
    return img_path.name


def load_labels_metadata(label_csv: Path,
                         raw_root: Path) -> Tuple[Dict[str, str], str, Dict[str, List[str]]]:
    """
    Loads labels.csv, validates its contents, and checks for ambiguous filenames.

    Supported CSV formats
    ---------------------
    1) filename,label
    2) relative_path,label

    Returns
    -------
    labels_map          : dict mapping key -> label
    key_column          : either 'filename' or 'relative_path'
    duplicate_filenames : duplicate basenames found in raw_root
    """
    if not Path(label_csv).exists():
        raise FileNotFoundError(
            f'Could not find label file: {label_csv}'
        )

    with open(label_csv, newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames or []

        if 'label' not in fieldnames:
            raise ValueError("labels.csv must contain a 'label' column.")

        if 'relative_path' in fieldnames:
            key_column = 'relative_path'
        elif 'filename' in fieldnames:
            key_column = 'filename'
        else:
            raise ValueError(
                "labels.csv must contain either a 'filename' column or a "
                "'relative_path' column."
            )

        labels_map = {}
        duplicates_in_csv = []

        for row_idx, row in enumerate(reader, start=2):  # header = row 1
            raw_key = row.get(key_column, '')
            key = str(raw_key).strip().replace('\\', '/')
            label = _normalise_label(row.get('label', ''))

            if not key:
                raise ValueError(
                    f'Empty {key_column} value found in labels.csv at row {row_idx}.'
                )

            if key in labels_map:
                duplicates_in_csv.append(key)
            else:
                labels_map[key] = label

    if duplicates_in_csv:
        preview = ', '.join(duplicates_in_csv[:5])
        raise ValueError(
            'Duplicate keys found in labels.csv. '
            f'Examples: {preview}. Each labelled image must appear only once.'
        )

    duplicate_filenames = _discover_duplicate_filenames(raw_root)

    if key_column == 'filename' and duplicate_filenames:
        preview_items = list(duplicate_filenames.items())[:5]
        preview_text = '\n'.join(
            f"  - {fname}: {paths}"
            for fname, paths in preview_items
        )
        raise ValueError(
            "Duplicate basenames were found in the raw dataset, so a "
            "'filename,label' CSV is ambiguous.\n"
            "Please switch to a 'relative_path,label' CSV instead.\n"
            f'Examples:\n{preview_text}'
        )

    print(f'Loaded {len(labels_map):,} labels using key column: {key_column}')
    if duplicate_filenames and key_column == 'relative_path':
        print(
            'Detected duplicate basenames in the raw data. '
            'Year-prefixed filenames will be used inside dataset/.'
        )

    return labels_map, key_column, duplicate_filenames


def build_dataset_structure(raw_root: Path,
                            dataset_root: Path,
                            label_csv: Path,
                            train_years: list,
                            val_years: list,
                            test_years: list,
                            rebuild: bool = True) -> None:
    """
    Restructures raw year-based image folders into a train/val/test directory tree
    suitable for torchvision.datasets.ImageFolder.

    Safety and reproducibility features
    -----------------------------------
    - Validates label values
    - Rejects ambiguous duplicate filenames
    - Deletes and rebuilds dataset/ from scratch when rebuild=True
    - Saves split manifests and skipped files for debugging
    """
    labels_map, key_column, duplicate_filenames = load_labels_metadata(
        label_csv=label_csv,
        raw_root=raw_root
    )

    all_years = train_years + val_years + test_years
    if len(all_years) != len(set(all_years)):
        raise ValueError('Train/val/test year lists must be disjoint.')

    split_map = {}
    for y in train_years:
        split_map[y] = 'train'
    for y in val_years:
        split_map[y] = 'val'
    for y in test_years:
        split_map[y] = 'test'

    if rebuild and dataset_root.exists():
        shutil.rmtree(dataset_root)
        print(f'Deleted existing dataset folder: {dataset_root}')

    for split in ['train', 'val', 'test']:
        for cls in ['waste', 'no_waste']:
            (dataset_root / split / cls).mkdir(parents=True, exist_ok=True)

    manifest_dir = dataset_root / 'manifests'
    manifest_dir.mkdir(parents=True, exist_ok=True)

    copied_rows = {split: [] for split in ['train', 'val', 'test']}
    skipped_rows = []

    copied_count = 0

    for year_folder in sorted(split_map.keys()):
        split = split_map[year_folder]
        year_path = raw_root / year_folder

        if not year_path.exists():
            print(f'Warning: {year_path} not found, skipping.')
            continue

        for img_path in _list_image_files(year_path):
            key = _label_key_for_path(img_path, raw_root, key_column)

            if key not in labels_map:
                skipped_rows.append({
                    'year_folder': year_folder,
                    'filename': img_path.name,
                    'relative_path': img_path.relative_to(raw_root).as_posix(),
                    'reason': 'missing_label'
                })
                continue

            cls = labels_map[key]
            copied_name = (
                f'{year_folder}__{img_path.name}'
                if duplicate_filenames else img_path.name
            )
            dest = dataset_root / split / cls / copied_name
            shutil.copy2(img_path, dest)

            copied_rows[split].append({
                'year_folder': year_folder,
                'split': split,
                'label': cls,
                'source_filename': img_path.name,
                'source_relative_path': img_path.relative_to(raw_root).as_posix(),
                'copied_filename': copied_name,
                'destination_relative_path': dest.relative_to(dataset_root).as_posix()
            })
            copied_count += 1

    for split, rows in copied_rows.items():
        manifest_path = manifest_dir / f'{split}_manifest.csv'
        with open(manifest_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(
                f,
                fieldnames=[
                    'year_folder', 'split', 'label',
                    'source_filename', 'source_relative_path',
                    'copied_filename', 'destination_relative_path'
                ]
            )
            writer.writeheader()
            writer.writerows(rows)

    skipped_path = manifest_dir / 'skipped_unlabelled.csv'
    with open(skipped_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(
            f,
            fieldnames=['year_folder', 'filename', 'relative_path', 'reason']
        )
        writer.writeheader()
        writer.writerows(skipped_rows)

    metadata = {
        'raw_root': str(raw_root),
        'label_csv': str(label_csv),
        'label_key_column': key_column,
        'rebuild': rebuild,
        'duplicate_basenames_in_raw_data': bool(duplicate_filenames),
        'train_years': train_years,
        'val_years': val_years,
        'test_years': test_years,
        'copied_images': copied_count,
        'skipped_unlabelled_images': len(skipped_rows)
    }
    with open(manifest_dir / 'build_metadata.json', 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)

    print(
        f'Dataset built successfully: {copied_count:,} images copied, '
        f'{len(skipped_rows):,} skipped (missing label).'
    )
    print(f'Manifests saved to: {manifest_dir}')
    _print_split_stats(dataset_root)


def _print_split_stats(dataset_root: Path) -> None:
    """Prints class counts for each split."""
    print('\nDataset statistics:')
    print(f'{"Split":<10} {"waste":>8} {"no_waste":>10} {"total":>8}')
    print('-' * 42)
    for split in ['train', 'val', 'test']:
        w = len(list((dataset_root / split / 'waste').glob('*')))
        nw = len(list((dataset_root / split / 'no_waste').glob('*')))
        print(f'{split:<10} {w:>8} {nw:>10} {w+nw:>8}')


# ── Run once after you have your label CSV ready ─────────────────────────────
# build_dataset_structure(
#     raw_root     = RAW_DATA_ROOT,
#     dataset_root = DATASET_ROOT,
#     label_csv    = LABEL_CSV,
#     train_years  = TRAIN_YEARS,
#     val_years    = VAL_YEARS,
#     test_years   = TEST_YEARS,
#     rebuild      = REBUILD_DATASET
# )
print('Dataset preparation function defined.')
print('Uncomment the call above once your labels.csv is ready.')


### Expected format for labels.csv

Preferred format when filenames are unique:
```csv
filename,label
pano_0000_001879_heading344_67_pitch-0_33_right90_square_fov90_0_640x640.jpg,no_waste
pano_0001_002345_heading120_45_pitch0_12_right90_square_fov90_0_640x640.jpg,waste
```

Safer format when duplicate basenames may exist across year folders:
```csv
relative_path,label
year_2016/pano_0000_001879_heading344_67_pitch-0_33_right90_square_fov90_0_640x640.jpg,no_waste
year_2021/pano_0001_002345_heading120_45_pitch0_12_right90_square_fov90_0_640x640.jpg,waste
```


## 4. Data Loaders & Augmentation

In [ ]:
def get_transforms(split: str) -> transforms.Compose:
    """
    Returns the appropriate torchvision transform pipeline for each split.

    Training  : resize + random augmentations + normalise
    Val / Test: resize + centre crop + normalise (no randomness)
    """
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.1
            ),
            transforms.RandomRotation(degrees=10),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])
    else:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.CenterCrop(IMG_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])


def _validate_split_class_balance(dataset_obj, split_name: str) -> None:
    class_names = dataset_obj.classes
    if set(class_names) != {'waste', 'no_waste'}:
        raise ValueError(
            f"{split_name} split must contain exactly the classes "
            f"'waste' and 'no_waste'. Found: {class_names}"
        )

    counts = np.bincount(dataset_obj.targets, minlength=len(class_names))
    stats = dict(zip(class_names, counts.tolist()))
    print(f'{split_name.title()} split class counts: {stats}')

    if split_name == 'train' and np.any(counts == 0):
        raise ValueError(
            'The training split is missing one of the classes. '
            'Both waste and no_waste are required for training.'
        )
    if split_name in {'val', 'test'} and np.any(counts == 0):
        print(
            f'Warning: {split_name} split is missing one of the classes. '
            'Some ranking metrics (e.g. Average Precision / ROC AUC) may be undefined.'
        )


def get_dataloaders(dataset_root: Path,
                    use_weighted_sampler: bool = USE_WEIGHTED_SAMPLER) -> dict:
    """
    Creates DataLoader objects for train, val, and test splits.

    By default, only ONE class-imbalance strategy is used: a
    WeightedRandomSampler on the training split. If disabled,
    the training loader falls back to ordinary shuffled batches.
    """
    datasets_dict = {
        split: datasets.ImageFolder(
            root=str(dataset_root / split),
            transform=get_transforms(split)
        )
        for split in ['train', 'val', 'test']
    }

    for split, dataset_obj in datasets_dict.items():
        _validate_split_class_balance(dataset_obj, split)

    train_dataset = datasets_dict['train']
    class_to_idx = train_dataset.class_to_idx
    positive_class_idx = class_to_idx['waste']

    train_loader_kwargs = {
        'dataset': datasets_dict['train'],
        'batch_size': BATCH_SIZE,
        'num_workers': NUM_WORKERS,
        'pin_memory': True
    }

    if use_weighted_sampler:
        class_counts = np.bincount(
            train_dataset.targets,
            minlength=len(train_dataset.classes)
        ).astype(float)
        class_weights = 1.0 / class_counts
        sample_weights = [class_weights[label] for label in train_dataset.targets]
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(train_dataset),
            replacement=True
        )
        train_loader_kwargs['sampler'] = sampler
        print('Training loader uses WeightedRandomSampler.')
    else:
        train_loader_kwargs['shuffle'] = True
        print('Training loader uses standard shuffled sampling.')

    loaders = {
        'train': DataLoader(**train_loader_kwargs),
        'val': DataLoader(
            datasets_dict['val'],
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True
        ),
        'test': DataLoader(
            datasets_dict['test'],
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True
        ),
        'class_names': train_dataset.classes,
        'class_to_idx': class_to_idx,
        'positive_class_idx': positive_class_idx,
        'negative_class_idx': class_to_idx['no_waste']
    }

    print('Class names (index order):', loaders['class_names'])
    print(f'Train batches : {len(loaders["train"])}')
    print(f'Val   batches : {len(loaders["val"])}')
    print(f'Test  batches : {len(loaders["test"])}')
    return loaders


# loaders = get_dataloaders(DATASET_ROOT)
print('DataLoader functions defined.')


### Visualise a batch of training images
Always worth checking that augmentations look sensible before training.

In [ ]:
def show_batch(loader: DataLoader, class_names: list, n: int = 8) -> None:
    """Displays n images from one batch with their class labels."""
    images, labels = next(iter(loader))
    images = images[:n]
    labels = labels[:n]

    # Undo ImageNet normalisation for display
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    images_disp = images * std + mean
    images_disp = images_disp.clamp(0, 1)

    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    for i, ax in enumerate(axes):
        img = images_disp[i].permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(class_names[labels[i].item()], fontsize=10)
        ax.axis('off')
    plt.suptitle('Sample training batch (after augmentation)', y=1.02)
    plt.tight_layout()
    plt.show()


# show_batch(loaders['train'], loaders['class_names'])   # Uncomment to use
print('Visualisation function defined.')

## 5. Model Definition

We load ResNet-50 with pretrained ImageNet weights, freeze all convolutional
layers, and replace the final fully-connected layer with a custom binary
classification head.

In [ ]:
def build_model(freeze_backbone: bool = True) -> nn.Module:
    """
    Builds a ResNet-50 model for binary waste classification.

    Architecture:
        ResNet-50 backbone (pretrained on ImageNet)
        └── Custom head:
            Linear(2048 → 256) → ReLU → Dropout(0.5) → Linear(256 → 1)

    The final layer outputs a single logit (no sigmoid here — sigmoid is
    applied inside BCEWithLogitsLoss for numerical stability).

    Parameters
    ----------
    freeze_backbone : If True, all ResNet layers except the custom head
                      are frozen (Phase 1 training).
    """
    # Load pretrained ResNet-50
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    # Freeze all backbone layers
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Replace the classification head
    # ResNet-50's final fc layer has in_features=2048
    in_features = model.fc.in_features   # 2048
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.5),
        nn.Linear(256, 1)                # Single logit for binary classification
    )
    # The new head's parameters are trainable by default

    model = model.to(DEVICE)

    # Count trainable vs total parameters
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total parameters    : {total:,}')
    print(f'Trainable parameters: {trainable:,}')

    return model


model = build_model(freeze_backbone=True)
print('\nModel built successfully.')

## 6. Training

The training loop handles:
- Forward pass and loss computation
- Backpropagation and weight updates
- Validation after each epoch
- Early stopping based on **validation Average Precision (AP)**
- Saving the best model checkpoint
- Logging ranking-oriented validation metrics:
  - Validation AP
  - Validation P@100


In [ ]:
def compute_class_weight(dataset_root: Path) -> torch.Tensor:
    """
    Computes the positive class weight for BCEWithLogitsLoss.

    This is optional and should not be combined with the weighted sampler
    unless you have evidence that both together improve results.
    """
    n_waste = len(list((dataset_root / 'train' / 'waste').glob('*')))
    n_no_waste = len(list((dataset_root / 'train' / 'no_waste').glob('*')))

    if n_waste == 0:
        raise ValueError('Cannot compute pos_weight because the train/waste folder is empty.')

    pos_weight = n_no_waste / n_waste
    print(f'Training set — waste: {n_waste}, no_waste: {n_no_waste}')
    print(f'BCEWithLogitsLoss pos_weight: {pos_weight:.2f}')
    return torch.tensor([pos_weight], dtype=torch.float32).to(DEVICE)


def _labels_to_binary(labels: torch.Tensor, positive_index: int) -> torch.Tensor:
    """Converts ImageFolder class indices to binary labels where waste=1."""
    return (labels == positive_index).float()


def _class_indices_from_binary(binary_preds: np.ndarray,
                               positive_index: int,
                               negative_index: int) -> np.ndarray:
    """Maps binary predictions back to ImageFolder class indices."""
    return np.where(binary_preds == 1, positive_index, negative_index)


def compute_precision_at_k_from_arrays(y_true_binary: np.ndarray,
                                       y_scores: np.ndarray,
                                       k: int = TOP_K) -> float:
    """Computes precision among the top-k highest-confidence predictions."""
    if len(y_true_binary) == 0:
        return float('nan')

    effective_k = min(k, len(y_true_binary))
    ranked_idx = np.argsort(-y_scores)[:effective_k]
    hits = y_true_binary[ranked_idx].sum()
    return float(hits / effective_k)


def compute_average_precision_safe(y_true_binary: np.ndarray,
                                   y_scores: np.ndarray) -> float:
    """Computes Average Precision, returning NaN if the metric is undefined."""
    positives = int(y_true_binary.sum())
    negatives = int((1 - y_true_binary).sum())

    if positives == 0 or negatives == 0:
        return float('nan')
    return float(average_precision_score(y_true_binary, y_scores))


def train_one_epoch(model,
                    loader,
                    criterion,
                    optimizer,
                    positive_index: int):
    """Runs one full pass over the training set."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        binary_labels = _labels_to_binary(labels, positive_index).unsqueeze(1).to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, binary_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == binary_labels).sum().item()
        total += images.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model,
             loader,
             criterion,
             positive_index: int,
             top_k: int = TOP_K):
    """
    Evaluates the model on a validation or test loader.

    Returns
    -------
    dict with loss, accuracy, average precision, and P@K.
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_scores = []
    all_true_binary = []

    for images, labels in loader:
        images = images.to(DEVICE)
        binary_labels = _labels_to_binary(labels, positive_index).unsqueeze(1).to(DEVICE)

        logits = model(images)
        loss = criterion(logits, binary_labels)
        probs = torch.sigmoid(logits)

        running_loss += loss.item() * images.size(0)
        preds = (probs >= 0.5).float()
        correct += (preds == binary_labels).sum().item()
        total += images.size(0)

        all_scores.extend(probs.squeeze(1).cpu().numpy())
        all_true_binary.extend(binary_labels.squeeze(1).cpu().numpy())

    y_scores = np.asarray(all_scores, dtype=float)
    y_true_binary = np.asarray(all_true_binary, dtype=int)

    metrics = {
        'loss': running_loss / total,
        'acc': correct / total,
        'ap': compute_average_precision_safe(y_true_binary, y_scores),
        'p_at_100': compute_precision_at_k_from_arrays(y_true_binary, y_scores, top_k)
    }
    return metrics


def train(model,
          loaders,
          num_epochs,
          lr,
          checkpoint_name,
          patience=PATIENCE,
          pos_weight=None,
          top_k: int = TOP_K):
    """
    Full training loop with validation monitoring and early stopping.

    Model selection is based on validation Average Precision (AP),
    which is more aligned with ranking quality than plain accuracy.
    """
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )

    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': [],
        'val_ap': [],
        'val_p_at_100': []
    }

    best_val_ap = float('-inf')
    best_checkpoint = CHECKPOINT_DIR / f'{checkpoint_name}.pth'
    epochs_no_improve = 0
    positive_index = loaders['positive_class_idx']

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=loaders['train'],
            criterion=criterion,
            optimizer=optimizer,
            positive_index=positive_index
        )
        val_metrics = evaluate(
            model=model,
            loader=loaders['val'],
            criterion=criterion,
            positive_index=positive_index,
            top_k=top_k
        )
        scheduler.step(val_metrics['loss'])

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_metrics['acc'])
        history['val_ap'].append(val_metrics['ap'])
        history['val_p_at_100'].append(val_metrics['p_at_100'])

        ap_text = 'n/a' if np.isnan(val_metrics['ap']) else f"{val_metrics['ap']:.4f}"
        print(
            f"Epoch [{epoch:>2}/{num_epochs}]  "
            f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
            f"Val Loss: {val_metrics['loss']:.4f}  Val Acc: {val_metrics['acc']:.4f}  "
            f"Val AP: {ap_text}  Val P@{top_k}: {val_metrics['p_at_100']:.4f}"
        )

        if np.isnan(val_metrics['ap']):
            raise ValueError(
                'Validation Average Precision is undefined because the validation set '
                'contains only one class. Please ensure val/ has both waste and no_waste.'
            )

        if val_metrics['ap'] > best_val_ap:
            best_val_ap = val_metrics['ap']
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_checkpoint)
            print(f'  ✓ New best model saved (val_ap={val_metrics["ap"]:.4f})')
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f'  Early stopping triggered after {epoch} epochs.')
                break

    print(f'\nTraining complete. Best val AP: {best_val_ap:.4f}')
    print(f'Best model saved to: {best_checkpoint}')
    return history


print('Training functions defined.')


## 7. Phase 1 — Train Classification Head Only

The ResNet backbone is frozen. Only the new linear head is updated.
This is fast and prevents the pretrained features from being destroyed
before the head has learned anything meaningful.

In [ ]:
# ── Uncomment and run after dataset and loaders are ready ────────────────────

# pos_weight = compute_class_weight(DATASET_ROOT) if USE_POS_WEIGHT else None

# model = build_model(freeze_backbone=True)

# history_phase1 = train(
#     model           = model,
#     loaders         = loaders,
#     num_epochs      = EPOCHS_HEAD,
#     lr              = LR_HEAD,
#     checkpoint_name = 'phase1_best',
#     patience        = PATIENCE,
#     pos_weight      = pos_weight,
#     top_k           = TOP_K
# )

print('Phase 1 training block ready. Uncomment to run.')


## 8. Phase 2 — Fine-Tune Backbone (Progressive Unfreezing)

Now the last ResNet block (`layer4`) is unfrozen alongside the head.
A lower learning rate is used to make small, careful adjustments to
the pretrained features rather than overwriting them.

In [ ]:
def unfreeze_layer4(model: nn.Module) -> nn.Module:
    """
    Unfreezes ResNet layer4 and the classification head for fine-tuning.
    All earlier layers remain frozen.
    """
    for param in model.parameters():
        param.requires_grad = False

    for param in model.layer4.parameters():
        param.requires_grad = True

    for param in model.fc.parameters():
        param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Trainable parameters after unfreezing layer4: {trainable:,}')
    return model


# ── Uncomment and run after Phase 1 ──────────────────────────────────────────

# # Load the best Phase 1 weights
# model.load_state_dict(
#     torch.load(CHECKPOINT_DIR / 'phase1_best.pth', map_location=DEVICE)
# )
# model = unfreeze_layer4(model)

# history_phase2 = train(
#     model           = model,
#     loaders         = loaders,
#     num_epochs      = EPOCHS_FINETUNE,
#     lr              = LR_FINETUNE,
#     checkpoint_name = 'phase2_best',
#     patience        = PATIENCE,
#     pos_weight      = pos_weight,
#     top_k           = TOP_K
# )

print('Phase 2 fine-tuning block ready. Uncomment to run.')


## 9. Training Curves

Plotting loss and accuracy curves is essential for diagnosing:
- **Overfitting**: training accuracy rises but validation accuracy drops
- **Underfitting**: both curves plateau at a poor accuracy
- **Good fit**: both curves converge to a similar, high accuracy

In [ ]:
def plot_training_curves(history_p1: dict, history_p2: dict = None) -> None:
    """
    Plots training and validation curves.

    Panels:
    - Loss
    - Accuracy
    - Ranking-oriented validation metrics (AP and P@100)
    """
    if history_p2 is not None:
        history = {k: history_p1[k] + history_p2[k] for k in history_p1}
        phase1_end = len(history_p1['train_loss'])
    else:
        history = history_p1
        phase1_end = None

    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(epochs, history['train_loss'], label='Train loss', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], label='Val loss', linewidth=2)
    if phase1_end:
        axes[0].axvline(phase1_end, color='grey', linestyle='--', label='Phase 2 starts')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss curves')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], label='Train acc', linewidth=2)
    axes[1].plot(epochs, history['val_acc'], label='Val acc', linewidth=2)
    if phase1_end:
        axes[1].axvline(phase1_end, color='grey', linestyle='--', label='Phase 2 starts')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy curves')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs, history['val_ap'], label='Val AP', linewidth=2)
    axes[2].plot(epochs, history['val_p_at_100'], label='Val P@100', linewidth=2)
    if phase1_end:
        axes[2].axvline(phase1_end, color='grey', linestyle='--', label='Phase 2 starts')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Score')
    axes[2].set_title('Ranking-oriented validation metrics')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.suptitle('Training curves — Waste Detection (ResNet-50)', fontsize=14)
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


# plot_training_curves(history_phase1, history_phase2)
print('Plot function defined.')


## 10. Evaluation on the Test Set

The test set is only evaluated **once**, after all training and tuning decisions
have been made using the validation set. This gives an unbiased estimate of
the model's real-world performance.

In [ ]:
@torch.no_grad()
def evaluate_test_set(model: nn.Module,
                      test_loader: DataLoader,
                      class_names: list,
                      top_k: int = TOP_K) -> None:
    """
    Full evaluation on the test set.

    Prints:
    - Classification report (precision, recall, F1 per class)
    - Confusion matrix
    - ROC curve with AUC score
    - Average Precision
    - P@100
    """
    model.eval()

    positive_index = class_names.index('waste')
    negative_index = class_names.index('no_waste')

    all_labels_idx = []
    all_probs = []

    for images, labels in test_loader:
        images = images.to(DEVICE)
        logits = model(images)
        probs = torch.sigmoid(logits).squeeze(1)

        all_labels_idx.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

    all_labels_idx = np.array(all_labels_idx)
    all_probs = np.array(all_probs)

    binary_labels = (all_labels_idx == positive_index).astype(int)
    binary_preds = (all_probs >= 0.5).astype(int)
    class_preds = _class_indices_from_binary(
        binary_preds=binary_preds,
        positive_index=positive_index,
        negative_index=negative_index
    )

    print('=' * 60)
    print('TEST SET EVALUATION')
    print('=' * 60)
    print(classification_report(
        all_labels_idx,
        class_preds,
        target_names=class_names
    ))

    test_ap = compute_average_precision_safe(binary_labels, all_probs)
    test_p_at_k = compute_precision_at_k_from_arrays(binary_labels, all_probs, top_k)
    print(f'Test Average Precision : {test_ap:.4f}' if not np.isnan(test_ap) else 'Test Average Precision : n/a')
    print(f'Test P@{min(top_k, len(binary_labels))}          : {test_p_at_k:.4f}')

    cm = confusion_matrix(all_labels_idx, class_preds)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title('Confusion Matrix — Test Set')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

    if binary_labels.sum() == 0 or binary_labels.sum() == len(binary_labels):
        print('ROC AUC is undefined because the test set contains only one class.')
        return

    fpr, tpr, _ = roc_curve(binary_labels, all_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve — Test Set')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
    plt.show()


# ── Load best model and evaluate ─────────────────────────────────────────────
# model.load_state_dict(
#     torch.load(CHECKPOINT_DIR / 'phase2_best.pth', map_location=DEVICE)
# )
# evaluate_test_set(model, loaders['test'], loaders['class_names'], top_k=TOP_K)

print('Test evaluation functions defined.')


## 11. Leaderboard — Top-100 Detections (P@100)

For the leaderboard, we run inference over the **entire 10,000-image dataset**,
collect the waste confidence score for every image, sort descending,
and take the top 100.

If labels are available, P@100 is computed by matching each prediction
back to `labels.csv` instead of inferring labels from folder names.


In [ ]:
from torch.utils.data import Dataset


class UnlabelledImageDataset(Dataset):
    """
    Loads all images from a directory tree without requiring labels.
    Used for full-dataset inference.
    """
    def __init__(self, root: Path, transform):
        self.paths = sorted(
            [
                p for p in root.rglob('*')
                if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
            ]
        )
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        img = self.transform(img)
        path = str(self.paths[idx])
        return img, path


def _lookup_label_for_inference_path(path: str,
                                     raw_root: Path,
                                     labels_map: Dict[str, str],
                                     key_column: str) -> str:
    img_path = Path(path)
    key = _label_key_for_path(img_path, raw_root, key_column)
    return labels_map.get(key)


def _compute_precision_at_100_from_labels(top_results: list,
                                          raw_root: Path,
                                          labels_map: Dict[str, str],
                                          key_column: str,
                                          top_k: int = TOP_K) -> None:
    """
    Computes P@K by reading labels from labels.csv.
    """
    inspected = top_results[:top_k]
    hits = 0
    labelled = 0

    for score, path in inspected:
        label = _lookup_label_for_inference_path(
            path=path,
            raw_root=raw_root,
            labels_map=labels_map,
            key_column=key_column
        )
        if label is None:
            continue

        labelled += 1
        hits += int(label == 'waste')

    if labelled == 0:
        print(f'\nP@{top_k} could not be computed: none of the top predictions have labels.')
        return

    if labelled < top_k:
        print(
            f'\nOfficial P@{top_k} could not be computed because '
            f'{top_k - labelled} of the top predictions are missing labels.'
        )
        print(f'Labelled subset precision = {hits}/{labelled} = {hits / labelled:.4f}')
        return

    precision = hits / top_k
    print(f'\nP@{top_k} = {hits}/{top_k} = {precision:.4f}')


@torch.no_grad()
def generate_top100(model: nn.Module,
                    raw_root: Path,
                    label_csv: Path = None,
                    output_dir: Path = Path('top100'),
                    top_k: int = TOP_K) -> None:
    """
    Runs inference on all images in raw_root, ranks them by waste confidence,
    copies the top-k images to output_dir, and optionally computes P@K if
    labels.csv is provided.
    """
    model.eval()
    output_dir.mkdir(parents=True, exist_ok=True)

    inference_transform = get_transforms('test')
    dataset = UnlabelledImageDataset(raw_root, inference_transform)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    print(f'Running inference on {len(dataset)} images...')

    results = []

    for images, paths in loader:
        images = images.to(DEVICE)
        logits = model(images)
        scores = torch.sigmoid(logits).squeeze(1)

        for score, path in zip(scores.cpu().numpy(), paths):
            results.append((float(score), path))

    results.sort(key=lambda x: x[0], reverse=True)

    labels_map = None
    key_column = None
    if label_csv is not None and Path(label_csv).exists():
        labels_map, key_column, _ = load_labels_metadata(label_csv=label_csv, raw_root=raw_root)

    scores_csv = output_dir / 'all_scores.csv'
    with open(scores_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['rank', 'score', 'filename', 'relative_path', 'label_if_available'])
        for rank, (score, path) in enumerate(results, start=1):
            rel_path = Path(path).relative_to(raw_root).as_posix()
            label = (
                _lookup_label_for_inference_path(path, raw_root, labels_map, key_column)
                if labels_map is not None else ''
            )
            writer.writerow([rank, f'{score:.6f}', Path(path).name, rel_path, label])
    print(f'All scores saved to {scores_csv}')

    print(f'\nTop {top_k} detections:')
    print(f'{"Rank":<6} {"Score":<10} {"Filename"}')
    print('-' * 80)

    for rank, (score, path) in enumerate(results[:top_k], start=1):
        fname = Path(path).name
        dest = output_dir / f'{rank:03d}_{fname}'
        shutil.copy2(path, dest)
        print(f'{rank:<6} {score:<10.4f} {fname}')

    print(f'\nTop {top_k} images copied to: {output_dir}')

    if labels_map is not None:
        _compute_precision_at_100_from_labels(
            top_results=results,
            raw_root=raw_root,
            labels_map=labels_map,
            key_column=key_column,
            top_k=top_k
        )
    else:
        print('\nNo labels.csv provided, so P@100 was not computed.')


# ── Run after loading best model ─────────────────────────────────────────────
# model.load_state_dict(
#     torch.load(CHECKPOINT_DIR / 'phase2_best.pth', map_location=DEVICE)
# )
# generate_top100(model, RAW_DATA_ROOT, label_csv=LABEL_CSV, top_k=TOP_K)

print('Leaderboard inference functions defined.')


## 12. Full Pipeline — End-to-End Run

Once your `labels.csv` is ready and the dataset folder has been built,
run this cell to execute the entire pipeline from start to finish.

In [ ]:
def run_full_pipeline():
    """
    Executes the complete pipeline:
    1. Build dataset structure from labels.csv
    2. Create data loaders
    3. Phase 1: train head only
    4. Phase 2: fine-tune with layer4 unfrozen
    5. Evaluate on test set
    6. Generate Top-100 leaderboard submission
    """
    print('STEP 1: Building dataset structure...')
    build_dataset_structure(
        raw_root=RAW_DATA_ROOT,
        dataset_root=DATASET_ROOT,
        label_csv=LABEL_CSV,
        train_years=TRAIN_YEARS,
        val_years=VAL_YEARS,
        test_years=TEST_YEARS,
        rebuild=REBUILD_DATASET
    )

    print('\nSTEP 2: Creating data loaders...')
    loaders = get_dataloaders(
        dataset_root=DATASET_ROOT,
        use_weighted_sampler=USE_WEIGHTED_SAMPLER
    )
    pos_weight = compute_class_weight(DATASET_ROOT) if USE_POS_WEIGHT else None

    print('\nSTEP 3: Phase 1 — training classification head...')
    model = build_model(freeze_backbone=True)
    history_p1 = train(
        model=model,
        loaders=loaders,
        num_epochs=EPOCHS_HEAD,
        lr=LR_HEAD,
        checkpoint_name='phase1_best',
        patience=PATIENCE,
        pos_weight=pos_weight,
        top_k=TOP_K
    )

    print('\nSTEP 4: Phase 2 — fine-tuning with layer4 unfrozen...')
    model.load_state_dict(
        torch.load(CHECKPOINT_DIR / 'phase1_best.pth', map_location=DEVICE)
    )
    model = unfreeze_layer4(model)
    history_p2 = train(
        model=model,
        loaders=loaders,
        num_epochs=EPOCHS_FINETUNE,
        lr=LR_FINETUNE,
        checkpoint_name='phase2_best',
        patience=PATIENCE,
        pos_weight=pos_weight,
        top_k=TOP_K
    )

    print('\nSTEP 5: Plotting training curves...')
    plot_training_curves(history_p1, history_p2)

    print('\nSTEP 6: Evaluating on test set...')
    model.load_state_dict(
        torch.load(CHECKPOINT_DIR / 'phase2_best.pth', map_location=DEVICE)
    )
    evaluate_test_set(model, loaders['test'], loaders['class_names'], top_k=TOP_K)

    print('\nSTEP 7: Generating leaderboard Top-100...')
    generate_top100(
        model=model,
        raw_root=RAW_DATA_ROOT,
        label_csv=LABEL_CSV,
        top_k=TOP_K
    )

    print('\nPipeline complete!')


# run_full_pipeline()
print('Full pipeline function defined. Uncomment run_full_pipeline() to execute.')


---
## Summary of Design Choices

| Decision | Choice | Justification |
|---|---|---|
| Architecture | ResNet-50 pretrained | Limited labelled data; ImageNet features transfer well |
| Input size | 224×224 | Standard for ResNet; original 640×640 is downscaled |
| Activation (output) | Implicit sigmoid in BCEWithLogitsLoss | Numerically stable |
| Loss function | BCEWithLogitsLoss | Clean binary objective; `pos_weight` kept optional and off by default |
| Imbalance handling | WeightedRandomSampler | Start with one imbalance strategy only |
| Dataset build | Rebuild from scratch + saved manifests | Reproducible and safe after relabelling |
| Label validation | Strict validation + duplicate filename check | Prevents silent data corruption |
| Optimiser | Adam | Robust adaptive learning rate |
| LR schedule | ReduceLROnPlateau on val loss | Reduces LR when optimisation stalls |
| Model selection | Best validation AP checkpoint | Better aligned with ranking than accuracy alone |
| Phase 1 | Frozen backbone, train head | Preserves pretrained features early |
| Phase 2 | Unfreeze layer4 + head | Fine-tunes high-level features for waste patterns |
| Early stopping | Patience=5 on val AP | Helps prevent overfitting |
| Validation metrics | Accuracy + AP + P@100 | Mix of threshold-based and ranking-based monitoring |
| Evaluation | Classification report + confusion matrix + ROC/AUC + AP + P@100 | More complete picture of performance |
| Leaderboard P@100 | Computed from `labels.csv` | Correct even when raw images are not inside `waste/` folders |
